# Day 58 — Handling Missing Values with Pipeline + ColumnTransformer

In [83]:
import pandas as pd
data = {
    'age': [25, 30, None, 40, 28, 32, 45, None, 38, 26],

    'salary': [40000, 50000, 60000, None, 45000,
               55000, 90000, 42000, None, 43000],

    'city': ['Delhi', 'Mumbai', 'Delhi', 'Bangalore',
             None, 'Delhi', 'Bangalore', 'Delhi',
             'Mumbai', None],

    'gender': ['Male', 'Female', 'Male', 'Female',
               'Male', None, 'Male', 'Female',
               'Male', 'Male'],

    'purchased': [0, 1, 1, 1, 0, 1, 1, 0, 1, 0]
}

df = pd.DataFrame(data)

print(df)

    age   salary       city  gender  purchased
0  25.0  40000.0      Delhi    Male          0
1  30.0  50000.0     Mumbai  Female          1
2   NaN  60000.0      Delhi    Male          1
3  40.0      NaN  Bangalore  Female          1
4  28.0  45000.0       None    Male          0
5  32.0  55000.0      Delhi    None          1
6  45.0  90000.0  Bangalore    Male          1
7   NaN  42000.0      Delhi  Female          0
8  38.0      NaN     Mumbai    Male          1
9  26.0  43000.0       None    Male          0


In [84]:
X = df.drop('purchased', axis = 1)

In [85]:
X

,age,salary,city,gender
0,25.0,40000.0,Delhi,Male
1,30.0,50000.0,Mumbai,Female
2,NaN,60000.0,Delhi,Male
3,40.0,NaN,Bangalore,Female
4,28.0,45000.0,None,Male
5,32.0,55000.0,Delhi,None
6,45.0,90000.0,Bangalore,Male
7,NaN,42000.0,Delhi,Female
8,38.0,NaN,Mumbai,Male
9,26.0,43000.0,None,Male


In [86]:
y = df['purchased']

In [87]:
y

0    0
1    1
2    1
3    1
4    0
5    1
6    1
7    0
8    1
9    0
Name: purchased, dtype: int64

In [88]:
num_cols = df.select_dtypes(include = ["int", "float"])
num_cols

,age,salary,purchased
0,25.0,40000.0,0
1,30.0,50000.0,1
2,NaN,60000.0,1
3,40.0,NaN,1
4,28.0,45000.0,0
5,32.0,55000.0,1
6,45.0,90000.0,1
7,NaN,42000.0,0
8,38.0,NaN,1
9,26.0,43000.0,0


In [89]:
num_cols = ['age', 'salary']
num_cols

['age', 'salary']

## The common error i was making that is i was taking the DataFrame like df[[]] and second is not using the .columns this are corrected below make you correct understanding and be careful form the further mistake

In [167]:
cat_cols = df.select_dtypes(include = ["object"])
cat_cols

,city,gender
0,Delhi,Male
1,Mumbai,Female
2,Delhi,Male
3,Bangalore,Female
4,None,Male
5,Delhi,None
6,Bangalore,Male
7,Delhi,Female
8,Mumbai,Male
9,None,Male


In [168]:
cat_cols = df.select_dtypes(include=["object"]).columns
cat_cols

Index(['city', 'gender'], dtype='object')

In [169]:
cat_cols = ["city", "gender"]

In [170]:
df.isnull().sum()

age          2
salary       2
city         2
gender       1
purchased    0
dtype: int64

In [171]:
from sklearn.impute import SimpleImputer

In [172]:
from sklearn.preprocessing import StandardScaler

In [173]:
from sklearn.pipeline import Pipeline

In [174]:
num_pipeline = Pipeline([
    ("Num", SimpleImputer(strategy = 'median')),
    ("Scale", StandardScaler())
])


In [175]:
num_pipeline

Pipeline(steps=[('Num', SimpleImputer(strategy='median')),
                ('Scale', StandardScaler())])

In [176]:
from sklearn.preprocessing import OneHotEncoder

In [177]:
cat_pipeline = Pipeline([
    ("Cat", SimpleImputer(strategy = 'most_frequent')),
    ("Encode", OneHotEncoder(handle_unknown = 'ignore'))
])

In [178]:
cat_pipeline

Pipeline(steps=[('Cat', SimpleImputer(strategy='most_frequent')),
                ('Encode', OneHotEncoder(handle_unknown='ignore'))])

In [179]:
from sklearn.compose import ColumnTransformer

In [180]:
preprocessor = ColumnTransformer([
    ("Num", num_pipeline, num_cols),
    ("Cat", cat_pipeline, cat_cols)
])

In [181]:
preprocessor

ColumnTransformer(transformers=[('Num',
                                 Pipeline(steps=[('Num',
                                                  SimpleImputer(strategy='median')),
                                                 ('Scale', StandardScaler())]),
                                 ['age', 'salary']),
                                ('Cat',
                                 Pipeline(steps=[('Cat',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('Encode',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['city', 'gender'])])

In [182]:
from sklearn.linear_model import LogisticRegression

In [183]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("estimators", LogisticRegression())
])
    

In [184]:
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('Num',
                                                  Pipeline(steps=[('Num',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scale',
                                                                   StandardScaler())]),
                                                  ['age', 'salary']),
                                                 ('Cat',
                                                  Pipeline(steps=[('Cat',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['city', 'gender'])])),
                ('estimators', LogisticRegression())])

In [185]:
from sklearn.model_selection import train_test_split

In [186]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [187]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('Num',
                                                  Pipeline(steps=[('Num',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scale',
                                                                   StandardScaler())]),
                                                  ['age', 'salary']),
                                                 ('Cat',
                                                  Pipeline(steps=[('Cat',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['city', 'gender'])])),
                ('estimators', LogisticRegression())])

In [188]:
y_pred = pipeline.predict(X_test)
y_pred

array([1, 0])

In [189]:
from sklearn.metrics import accuracy_score 

In [190]:
accuracy_score(y_pred, y_test)

0.5